# Step 5 - Base ranking

Scores and rankings of every method on the **original** graph. These are the reference that the perturbed rankings will be compared to in step 8.

Methods: the two forests (with / without PageRank input) and the five single centralities (baselines).

In [1]:
import os, sys
if 'google.colab' in sys.modules and not os.path.exists('Final-Project'):
    !git clone https://github.com/ShaharBar99/Final-Project.git
    %cd Final-Project
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

## 1. Load data and models

If the saved forests are missing (they are not on GitHub), they are rebuilt with the same seed.

In [2]:
import pandas as pd
from src import config, model, ranking

x = pd.read_csv(config.PROCESSED_DIR / 'features_scaled.csv', index_col='paper_id')
y = x[model.TARGET]
variants = {'with_pagerank': model.FEATURES_WITH_PAGERANK,
            'without_pagerank': model.FEATURES_WITHOUT_PAGERANK}
models_dir = config.PROCESSED_DIR / 'models'
forests = {name: model.load_or_train(name, x[cols], y, models_dir) for name, cols in variants.items()}

## 2. Scores and rankings

In [3]:
scores = ranking.base_scores(x, forests)
ranks = scores.apply(ranking.to_ranking)
ranks.head()

,rf_with_pagerank,rf_without_pagerank,degree,betweenness,closeness,pagerank,kcore
paper_id,,,,,,,
19127292,18694.0,18700.0,469.0,10762.5,18694.0,18694.0,2100.0
17363749,11068.0,7208.0,8945.0,10762.5,8779.5,11068.0,8436.5
19668377,18694.0,18700.0,580.5,10762.5,18694.0,18694.0,1253.5
17293876,16.0,19.0,34.5,10762.5,657.0,16.0,299.0
1313726,9643.0,5939.0,650.5,798.0,5954.0,9643.0,3416.0


## 3. Ties

Many papers share exactly the same score (for example all papers with betweenness 0). They get the average rank, so no fake order is invented between them. Methods with a lot of ties are limited in how finely they can rank at all.

In [4]:
ties = pd.DataFrame({
    'share of papers in a tie': scores.apply(ranking.tie_share),
    'distinct scores': scores.nunique(),
}).round(3)
ties

,share of papers in a tie,distinct scores
rf_with_pagerank,0.652,8305
rf_without_pagerank,0.793,4684
degree,0.999,82
betweenness,0.942,1270
closeness,0.884,2816
pagerank,0.660,7967
kcore,1.000,10


## 4. How similar are the methods to each other?

Spearman correlation between the base rankings.

In [5]:
agreement = ranks.corr(method='spearman').round(2)
agreement

,rf_with_pagerank,rf_without_pagerank,degree,betweenness,closeness,pagerank,kcore
rf_with_pagerank,1.00,0.81,0.29,0.19,0.73,1.00,0.37
rf_without_pagerank,0.81,1.00,0.39,0.23,0.88,0.81,0.48
degree,0.29,0.39,1.00,0.46,0.31,0.29,0.97
betweenness,0.19,0.23,0.46,1.00,0.16,0.19,0.42
closeness,0.73,0.88,0.31,0.16,1.00,0.73,0.41
pagerank,1.00,0.81,0.29,0.19,0.73,1.00,0.37
kcore,0.37,0.48,0.97,0.42,0.41,0.37,1.00


In [6]:
top = 100
overlap = pd.DataFrame({a: {b: len(set(ranks[a].nsmallest(top).index) & set(ranks[b].nsmallest(top).index))
                            for b in ranks.columns} for a in ranks.columns})
print(f'papers shared by the top {top} of each pair of methods:')
overlap

papers shared by the top 100 of each pair of methods:


,rf_with_pagerank,rf_without_pagerank,degree,betweenness,closeness,pagerank,kcore
rf_with_pagerank,100,65,26,3,35,100,15
rf_without_pagerank,65,100,33,8,46,65,26
degree,26,33,100,21,14,26,19
betweenness,3,8,21,100,4,3,26
closeness,35,46,14,4,100,35,24
pagerank,100,65,26,3,35,100,15
kcore,15,26,19,26,24,15,100


## 5. Save

In [7]:
scores.to_csv(config.PROCESSED_DIR / 'base_scores.csv', index_label='paper_id')
ranks.to_csv(config.PROCESSED_DIR / 'base_ranking.csv', index_label='paper_id')
print('saved', ranks.shape)

saved (19717, 7)
